# Synthetic Data Generator

This project generates realistic synthetic data for analytics practice, SQL testing, and portfolio projects.

Features:
- Thai demographic customer profiles
- Prefixed readable Entity identifiers
- Weighted regional distribution
- Realistic age generation 

## Imports

In [1]:
from faker import Faker 
import datetime
import random
import pandas as pd

## Generate Synthetic Dataset

In [2]:
def create_synthetic_data(n_rows):
    
    fake = Faker('en_TH')
    # seed 92 is initialized to guarantee deterministic record generation, ensuring permanent identifier consistency for entity-relationship diagrams (ERDs) and database joins.
    SEED = 92
    random.seed(SEED)
    fake.seed_instance(SEED)

    Customer_ID = []
    Customer_Name = []
    Email = []
    Phone = []
    Address = []
    District = []
    City = []
    Postal = []
    Region = []
    Date_of_Birth = []
    Gender = []
    Status = []
    Created_At = []

    for i in range(n_rows):
        
        # District entity prefixes are implemented to prevent identifier collision during table joins. 
        prefix_ID = "CUST_"
        Customer_ID.append(prefix_ID + fake.uuid4()[:4])

        # To keep the data looking realistic, `Customer_Names` are decided by gender.
        selected_gender = random.choice(['Male', 'Female', 'Other'])
        Gender.append(selected_gender)
        if selected_gender == 'Male':
            Customer_Name.append(fake.name_male())
        elif selected_gender == 'Female':
            Customer_Name.append(fake.name_female())
        else:
            Customer_Name.append(fake.name())

        Email.append(fake.unique.free_email())

        # force the phone numbers to start with `06`, `08`, or `09` so they look like actual Thai mobile numbers.
        prefix_phone = random.choice(['06', '08', '09'])
        phone = fake.unique.numerify(prefix_phone + '-####-####')
        Phone.append(phone)

        Address.append(fake.street_address())

        District.append(fake.city())

        City.append(fake.city())

        Postal.append(fake.postcode())

        # set custom percentages for customer regions and weigh more for Central and Northern where `Bangkok` and `Chiang Mai` are located.
        Region.append(random.choices(['Northern', 'Northeast', 'Southern', 'Eastern', 'Western', 'Central'], weights=[0.3, 0.15, 0.05, 0.05, 0.05, 0.4])[0])

        # center the ages around 30 years old and std for 7 years to keep the customer base realistic, then calculate the matching date of birth format.
        random_age = int(random.gauss(30, 7))
        random_age = max(18, min(random_age, 70))
        birth_year = datetime.date.today().year - random_age
        random_dob = datetime.date(birth_year, random.randint(1, 12), random.randint(1, 28))
        Date_of_Birth.append(random_dob.strftime('%Y-%m-%d'))

        Status.append(random.choice(['Single', 'Married', 'Divorced', 'Widowed', 'Other']))

        # To simulate customer acquisition momentum, the `Created_At` dates are generated based on Thailand's GDP path from 2021 to 2026. 
        cus_acq = [
        2021,
        2022,
        2023,
        2024,
        2025,
        2026 
        ] 
        cus_acq_weights = [
        0.10, 
        0.12,
        0.23, 
        0.28, 
        0.17, 
        0.10
        ]
        signup_year = random.choices(cus_acq, weights=cus_acq_weights)[0]
        # For 2026, the date range is limited to January 1st to May 28th to reflect the current date and maintain data realism.
        if signup_year == 2026: 
            signup_date = random.randint(1, 5), random.randint(1, 28)
        else:
            signup_date = random.randint(1, 12), random.randint(1, 28)
        random_created_at = datetime.date(signup_year, signup_date[0], signup_date[1])
        Created_At.append(random_created_at.strftime('%Y-%m-%d'))
    
    Customer_Data = pd.DataFrame({
        'Customer_ID': Customer_ID,
        'Customer_Name': Customer_Name,
        'Email': Email,
        'Phone': Phone,
        'Address': Address,
        'District': District,
        'City': City,
        'Postal': Postal,
        'Region': Region,
        'Date_of_Birth': Date_of_Birth,
        'Gender': Gender,
        'Status': Status,
        'Created_At': Created_At
    })
    return Customer_Data

## Preview

In [3]:
synthetic_customer_data = create_synthetic_data(5000)
synthetic_customer_data

,Customer_ID,Customer_Name,Email,Phone,Address,District,City,Postal,Region,Date_of_Birth,Gender,Status,Created_At
0,CUST_8cd2,Chaiwut Pasuk,eboonpungbaramee@hotmail.com,09-5000-2929,672 Tunradee Turnpike,Sujjaboriboonchester,Phenphitchaton,36273,Western,2005-11-25,Female,Single,2024-08-06
1,CUST_2df5,Kamolchanok Bunlupong,jitrinprachayaroch@yahoo.com,09-4441-9805,99451 Pitipat Overpass Apt. 452,Noppakaoside,Kongchayasukawutburgh,42110,Central,2000-06-02,Male,Single,2025-11-21
2,CUST_239e,Nutwadee Polauaypon,pianduangsrijaruwan@hotmail.com,09-5810-8166,357 Turongkinanon Extension,Port Phenphitcha,Benchapatranonshire,59801,Northern,2005-08-24,Male,Married,2024-11-28
3,CUST_a311,Jaruwan Pothanun,thanatchatrikasemmart@gmail.com,08-1460-3382,791 Pawan Summit Suite 266,Nutkritaton,Phubeschester,81771,Northern,2004-06-14,Male,Single,2023-03-23
4,CUST_9781,CPL Atit Kitprapa,supasita59@hotmail.com,06-3505-2396,7737 Chowitunkit Courts Apt. 411,Port Chayapat,Chayaninview,89080,Western,1983-03-16,Male,Single,2023-05-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,CUST_04d7,Pannawich Polauaypon,peemboonpungbaramee@gmail.com,06-5205-7744,03534 Jitrin Fields,New Chaiwut,North Anon,53558,Central,1995-11-21,Female,Married,2026-02-13
4996,CUST_478d,Sitiwat Prayoonhong,chaihirankarnpatchaploy@gmail.com,08-1200-0070,41229 Chaifah Branch,Wasununberg,Port Arisara,12950,Eastern,1991-08-15,Other,Married,2025-01-19
4997,CUST_6622,Apisara Sooksawang,patchaployturongkinanon@hotmail.com,08-7823-2469,752 Patcharaporn Isle Suite 869,Sarunpornbury,Chaifahtown,66678,Central,1993-12-18,Male,Married,2022-02-03
4998,CUST_ebf8,Kamolchanok Bunlerngsri,stitipatrayunyong@yahoo.com,08-5091-6503,472 Pattamon Mills,Matinawinshire,Peemberg,48316,Central,1994-08-08,Female,Single,2024-07-07
